In [ ]:
# Configuracion de cluster Slurm (alineada con run.sh / run_parallel.sh)
SLURM_PARTITION = 'long'
SLURM_CPUS_PER_TASK = 8
SLURM_MEM = '32G'
SLURM_GRES_EXPERIMENT = 'shard:4'
SLURM_GRES_MASSIVE = 'shard:6'
CONDA_ENV = 'RFA2526pt'
PYTORCH_CUDA_ALLOC_CONF = 'expandable_segments:True'

import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = PYTORCH_CUDA_ALLOC_CONF

print('Partition:', SLURM_PARTITION)
print('CPUs:', SLURM_CPUS_PER_TASK, '| Mem:', SLURM_MEM)
print('GRES exp:', SLURM_GRES_EXPERIMENT, '| GRES masivo:', SLURM_GRES_MASSIVE)
print('Conda env:', CONDA_ENV)
print('PYTORCH_CUDA_ALLOC_CONF:', os.environ['PYTORCH_CUDA_ALLOC_CONF'])


# 06 - Experimento Video Clean + CLIP

Objetivo tecnico:
1. Eliminar frames negros, de transicion brusca y practicamente estaticos.
2. Extraer embeddings visuales robustos con CLIP (en vez de estadisticas ligeras).
3. Entrenar regresion logistica para ES, EN y ES+EN.
4. Exportar tres JSON sobre el conjunto completo disponible.


In [ ]:
import json
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import cv2
from PIL import Image

import torch
from tqdm.auto import tqdm

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from transformers import AutoModel, AutoProcessor

def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd] + list(cwd.parents)
    for c in candidates:
        if (c / 'materials' / 'dataset_task3_exist2026').exists() and (c / 'notebooks_shiyi').exists():
            return c / 'notebooks_shiyi'
    if (cwd / 'artifacts').exists() or (cwd / 'entregables').exists():
        return cwd
    raise FileNotFoundError('Could not resolve project root with materials/dataset_task3_exist2026.')

def resolve_train_json(project_root: Path) -> Path:
    local_candidate = project_root.parent / 'materials' / 'dataset_task3_exist2026' / 'training.json'
    cluster_candidate = Path('/home/alumno.upv.es/scheng1/EXIST 2026 Videos Dataset/training/EXIST2026_training.json')
    if local_candidate.exists():
        return local_candidate
    if cluster_candidate.exists():
        return cluster_candidate
    raise FileNotFoundError('Training JSON not found in local materials or cluster path.')

PROJECT_ROOT = resolve_project_root()
DATA_JSON_PATH = resolve_train_json(PROJECT_ROOT)
VIDEO_ROOT = DATA_JSON_PATH.parent
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
DELIVERABLES_DIR = PROJECT_ROOT / 'entregables'
CACHE_NPZ = ARTIFACTS_DIR / 'video_clean_clip_features.npz'

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
DELIVERABLES_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
print('PROJECT_ROOT:', PROJECT_ROOT)
print('DATA_JSON_PATH:', DATA_JSON_PATH)

In [ ]:
with open(DATA_JSON_PATH, 'r', encoding='utf-8') as f:
    raw = json.load(f)

rows = []
for sid, obj in raw.items():
    rec = {'sample_id': str(sid)}
    rec.update(obj)
    rows.append(rec)

df = pd.DataFrame(rows)

def majority_task3_1(lbls):
    vals = [x for x in lbls if x in {'YES', 'NO'}]
    if not vals:
        return 'UNKNOWN'
    c = Counter(vals)
    if c['YES'] == c['NO']:
        return vals[0]
    return c.most_common(1)[0][0]

df['label'] = df['labels_task3_1'].apply(majority_task3_1)
df['y'] = df['label'].map({'NO': 0, 'YES': 1}).fillna(-1).astype(int)
df['video_path_abs'] = df['path_video'].apply(lambda p: str((VIDEO_ROOT / p).resolve()))

df = df.sort_values('sample_id').reset_index(drop=True)
print('n=', len(df), '| langs=', df['lang'].value_counts().to_dict())


In [ ]:
# Diseno de limpieza temporal:
# - frame negro: intensidad media muy baja.
# - frame estatico: diferencia media muy baja respecto al ultimo frame aceptado.
# - frame de transicion brusca: diferencia muy alta (flash/corte).
def clean_frames_from_video(video_path, sample_step=6, max_frames=12,
                            black_thr=12.0, static_thr=2.0, transition_thr=55.0):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return []

    selected = []
    prev_kept_gray = None
    idx = 0

    while True:
        ok, frame = cap.read()
        if not ok:
            break

        if idx % sample_step != 0:
            idx += 1
            continue

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        intensity = float(gray.mean())
        if intensity < black_thr:
            idx += 1
            continue

        if prev_kept_gray is not None:
            diff = np.mean(np.abs(gray.astype(np.float32) - prev_kept_gray.astype(np.float32)))
            if diff < static_thr:
                idx += 1
                continue
            if diff > transition_thr:
                idx += 1
                continue

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        selected.append(Image.fromarray(rgb))
        prev_kept_gray = gray
        idx += 1

        if len(selected) >= max_frames:
            break

    cap.release()
    return selected


In [ ]:
clip_model_name = 'openai/clip-vit-base-patch32'
clip_processor = AutoProcessor.from_pretrained(clip_model_name)
clip_model = AutoModel.from_pretrained(clip_model_name).to(DEVICE)
clip_model.eval()

def encode_images_clip(frames):
    inputs = clip_processor(images=frames, return_tensors='pt', padding=True)
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    with torch.no_grad():
        out = clip_model(**inputs)
        # Robust extraction: try pooled output first, then CLS token fallback.
        if hasattr(out, 'pooler_output') and out.pooler_output is not None:
            feats = out.pooler_output
        elif hasattr(out, 'last_hidden_state') and out.last_hidden_state is not None:
            feats = out.last_hidden_state[:, 0, :]
        else:
            feats = out[0][:, 0, :]
        feats = torch.nn.functional.normalize(feats, dim=-1)
    return feats

if CACHE_NPZ.exists():
    cache = np.load(CACHE_NPZ, allow_pickle=True)
    X_video = cache['X_video']
    sid_cache = cache['sample_ids'].astype(str)
    assert list(sid_cache) == df['sample_id'].astype(str).tolist()
    print('Loaded cache:', CACHE_NPZ, '| shape=', X_video.shape)
else:
    embs = []
    for row in tqdm(df.itertuples(index=False), total=len(df), desc='Video clean + CLIP'):
        frames = clean_frames_from_video(row.video_path_abs)
        if len(frames) == 0:
            frames = [Image.new('RGB', (224, 224), color=(0, 0, 0))]

        feats = encode_images_clip(frames)
        emb = feats.mean(dim=0).cpu().numpy()
        embs.append(emb)

    X_video = np.vstack(embs).astype(np.float32)
    np.savez_compressed(CACHE_NPZ, X_video=X_video, sample_ids=df['sample_id'].astype(str).values)
    print('Saved cache:', CACHE_NPZ, '| shape=', X_video.shape)

In [ ]:
def train_and_export_three_lang_models(X, y, langs, sample_ids, exp_tag):
    configs = {
        'ES': ['es'],
        'EN': ['en'],
        'ES_EN': ['es', 'en'],
    }

    for cfg_name, cfg_langs in configs.items():
        m = np.isin(langs, cfg_langs) & (y >= 0)
        Xtr, ytr = X[m], y[m]

        clf = Pipeline([
            ('scaler', StandardScaler()),
            ('lr', LogisticRegression(max_iter=2500, class_weight='balanced', n_jobs=-1)),
        ])
        clf.fit(Xtr, ytr)

        pred = clf.predict(X)
        pred_labels = np.where(pred == 1, 'YES', 'NO')

        out = [
            {'test_case': 'EXIST2026', 'id': str(sid), 'value': str(lbl)}
            for sid, lbl in zip(sample_ids, pred_labels)
        ]
        out_path = DELIVERABLES_DIR / f'BeingChillingWeWillWin_{exp_tag}_{cfg_name}.json'
        with open(out_path, 'w', encoding='utf-8') as f:
            json.dump(out, f, ensure_ascii=False)
        print('Saved', out_path, '| rows=', len(out))


train_and_export_three_lang_models(
    X_video,
    df['y'].to_numpy(np.int64),
    df['lang'].astype(str).to_numpy(),
    df['sample_id'].astype(str).to_numpy(),
    exp_tag='06VideoClean_CLIP',
)
